# Model Inference — final submission


In [ ]:
%pip install -q seaborn mlflow dagshub lightgbm xgboost

In [ ]:
import os, gc, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow

import preprocessing as prep
import evaluation as ev
from preprocessing import BASE_COLS, MD_COLS, WalmartFeatureBuilder, \
    feature_columns, log1p_clip, expm1_inv, calendar_frame
from evaluation import wmae

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
sns.set_style("whitegrid")
SEED = 42
np.random.seed(SEED)
os.makedirs("pictures", exist_ok=True)

CODE_PATHS = ["preprocessing.py", "evaluation.py"]

ARCH = "Inference"

In [ ]:
import dagshub

DAGSHUB_USER = "rkvit23"
DAGSHUB_REPO = "ML-FINAL"

dagshub.init(repo_owner=DAGSHUB_USER, repo_name=DAGSHUB_REPO, mlflow=True)
mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow")

EXPERIMENT_NAME = f"{ARCH}_Training"
mlflow.set_experiment(EXPERIMENT_NAME)
print("MLflow experiment:", EXPERIMENT_NAME)

In [ ]:
train_raw, test_raw, features_raw, stores_raw = prep.load_data()

TRAIN_START, TRAIN_END = train_raw.Date.min(), train_raw.Date.max()
TEST_START,  TEST_END  = test_raw.Date.min(),  test_raw.Date.max()
HORIZON = test_raw.Date.nunique()

print("train:", train_raw.shape, TRAIN_START.date(), "->", TRAIN_END.date(),
      "| weeks:", train_raw.Date.nunique())
print("test :", test_raw.shape,  TEST_START.date(),  "->", TEST_END.date(),
      "| weeks:", HORIZON)
print("series (Store, Dept) in train:", train_raw.groupby(["Store", "Dept"]).ngroups)

## Load the model from the Model Registry

In [ ]:
MODEL_NAME = "WalmartBestModel"
MODEL_VERSION = "latest"

client = mlflow.tracking.MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
print("Versions in the Registry:")
for v in versions:
    print(f"  v{v.version} | run_id={v.run_id} | {v.source}")

if MODEL_VERSION == "latest":
    MODEL_VERSION = max(int(v.version) for v in versions)
MODEL_URI = f"models:/{MODEL_NAME}/{MODEL_VERSION}"
print("\nloading:", MODEL_URI)

model = mlflow.pyfunc.load_model(MODEL_URI)
print("model loaded:", type(model))

In [ ]:
t0 = time.time()
test_pred = np.asarray(model.predict(test_raw[["Store", "Dept", "Date", "IsHoliday"]]))
print(f"predicted {len(test_pred):,} rows in {time.time()-t0:.1f}s")
assert len(test_pred) == len(test_raw) and np.isfinite(test_pred).all()

sub = ev.make_submission(test_raw, test_pred, "submission.csv")
print(sub.head())
print("saved: submission.csv")

In [ ]:
APPLY_XMAS_SHIFT = True

if APPLY_XMAS_SHIFT:
    sub_shift = ev.apply_christmas_shift(sub, test_raw)
    sub_shift.to_csv("submission_xmas_shift.csv", index=False)
    print("saved: submission_xmas_shift.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
hist_tot = train_raw[train_raw.Date >= "2011-11-01"].groupby("Date").Weekly_Sales.sum() / 1e6
pred_tot = pd.Series(test_pred, index=test_raw.Date).groupby(level=0).sum() / 1e6
ax.plot(hist_tot.index, hist_tot.values, label="train (actual)")
ax.plot(pred_tot.index, pred_tot.values, "--", label="test (forecast)")
ax.axvline(TRAIN_END, color="gray", ls=":")
ax.set_title("Total weekly sales: history + final forecast ($M)")
ax.legend()
plt.tight_layout(); plt.savefig("pictures/final_forecast.png", dpi=120); plt.show()